# 🎙️ Piper TTS: Персональный офлайн-голос для Android

**Инструкция: Нажмите `Среда выполнения` -> `Выполнить все` (Runtime -> Run all).**

Обучение выполнится полностью автоматически:
- Создается изолированная среда Python 3.10 с PyTorch.
- Загружается датасет из 50 фраз.
- Модель дообучается на вашем дикторском голосе.
- Готовый файл `my_voice.onnx` скачивается на устройство.

## Шаг 1. Подготовка изолированного окружения Python 3.10 и зависимостей

In [ ]:
# Проверка GPU
!nvidia-smi || echo 'Запуск на стандартном ускорителе'

# Системные пакеты
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg build-essential git

# Установка uv и создание Python 3.10 venv (с авто-очисткой --clear, без интерактивных запросов)
!pip install -q uv
!uv venv --clear --seed --python 3.10 /content/env

# Сверхбыстрая установка PyTorch
!uv pip install --python /content/env/bin/python torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Установка совместимых версий библиотек (включая setuptools<70 и numpy<2)
!uv pip install --python /content/env/bin/python "setuptools<70" "numpy<2" pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize==1.1.0 'cython<3' onnx onnxruntime soundfile librosa piper-tts

# Клонирование репозитория Piper
import os
if not os.path.exists('/content/piper'):
    !git clone --depth 1 https://github.com/rhasspy/piper.git /content/piper

# Сборка C-расширения monotonic_align для VITS
!cd /content/piper/src/python/piper_train/vits/monotonic_align && mkdir -p monotonic_align && /content/env/bin/cythonize -i core.pyx && mv core*.so monotonic_align/

# Установка пакетов piper в venv
!uv pip install --python /content/env/bin/python --no-deps -e /content/piper/src/python_run
!uv pip install --python /content/env/bin/python --no-deps -e /content/piper/src/python

# Проверка работоспособности
!/content/env/bin/python -c "import torch, piper_train, piper_phonemize, pytorch_lightning; from piper_train.vits.monotonic_align import maximum_path; print('✓ CUDA активна:', torch.cuda.is_available()); print('✓ Окружение Python 3.10 и C-расширения полностью готовы!')"


## Шаг 2. Загрузка вашего датасета и базовой модели Dmitri

In [ ]:
import os, shutil

# Загружаем свежий датасет из репозитория
if not os.path.exists('/content/piper-voice'):
    !git clone https://github.com/Holki9/piper-voice.git /content/piper-voice
else:
    !cd /content/piper-voice && git pull

# Копируем датасет в рабочую папку
!rm -rf /content/dataset
!cp -r /content/piper-voice/dataset /content/dataset
print('✓ Датасет готов: 102 сэмпла со студийным качеством и выравниванием Whisper')

# Скачиваем базовый русский чекпоинт Dmitri medium
!mkdir -p /content/base_model
!wget -q -c -O /content/base_model/dmitri.ckpt https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/epoch%3D5589-step%3D1478840.ckpt
!wget -q -c -O /content/base_model/config.json https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/ru/ru_RU/dmitri/medium/config.json
print('✓ Базовый чекпоинт Dmitri medium загружен!')

## Шаг 3. Предобработка датасета (фонетизация)

In [ ]:
# Очищаем предыдущий кэш обучения
!rm -rf /content/training_dir

!/content/env/bin/python -m piper_train.preprocess \
    --language ru \
    --input-dir /content/dataset \
    --output-dir /content/training_dir \
    --dataset-format ljspeech \
    --single-speaker \
    --sample-rate 22050 \
    --max-workers 2

print('✓ Предобработка успешно завершена!')
!head -n 5 /content/training_dir/dataset.jsonl

## Шаг 4. Запуск дообучения (Fine-Tuning)

In [ ]:
import subprocess

# Считываем точную эпоху чекпоинта через изолированное окружение
epoch_cmd = "import torch; c = torch.load('/content/base_model/dmitri.ckpt', map_location='cpu'); print(c.get('epoch', 5589))"
base_epoch = int(subprocess.check_output(['/content/env/bin/python', '-c', epoch_cmd]).decode().strip())
extra_epochs = 400
target_max_epochs = base_epoch + extra_epochs

# Проверяем наличие GPU
gpu_check = "import torch; print('gpu' if torch.cuda.is_available() else 'cpu')"
accel = subprocess.check_output(['/content/env/bin/python', '-c', gpu_check]).decode().strip()

print(f'Базовая эпоха чекпоинта: {base_epoch}')
print(f'Целевая конечная эпоха max_epochs: {target_max_epochs} (+{extra_epochs} эпох дообучения)')
print(f'Используемый ускоритель: {accel.upper()}')

# Запуск обучения
!/content/env/bin/python -m piper_train \
    --dataset-dir /content/training_dir \
    --accelerator {accel} \
    --devices 1 \
    --batch-size 8 \
    --validation-split 0.0 \
    --num-test-examples 0 \
    --max_epochs {target_max_epochs} \
    --checkpoint-epochs 50 \
    --precision 32 \
    --quality medium \
    --resume_from_checkpoint /content/base_model/dmitri.ckpt

## Шаг 5. Экспорт в ONNX и тестирование голоса

In [ ]:
# Скрипт безопасного экспорта
export_code = '''
import glob, os, shutil, subprocess

ckpts = sorted(glob.glob('/content/training_dir/lightning_logs/**/checkpoints/*.ckpt', recursive=True), key=os.path.getmtime)
if not ckpts:
    ckpts = sorted(glob.glob('/content/training_dir/*.ckpt'), key=os.path.getmtime)

best_ckpt = ckpts[-1]
print(f'Экспорт финального чекпоинта: {best_ckpt}')

subprocess.run(['/content/env/bin/python', '-m', 'piper_train.export_onnx', best_ckpt, '/content/my_voice.onnx'], check=True)
shutil.copy('/content/training_dir/config.json', '/content/my_voice.onnx.json')

test_text = 'Привет! Это мой персональный дикторский голос. Теперь я говорю полностью офлайн прямо на вашем устройстве.'
subprocess.run(
    f"echo '{test_text}' | /content/env/bin/piper --model /content/my_voice.onnx --config /content/my_voice.onnx.json --output_file /content/test.wav",
    shell=True, check=True
)
print('✓ Экспорт и тестовый синтез успешно завершены!')
'''
with open('/content/export_script.py', 'w', encoding='utf-8') as f:
    f.write(export_code)

!/content/env/bin/python /content/export_script.py

# Воспроизведение тестового сэмпла в блокноте
from IPython.display import Audio, display
print('Слушайте результат синтеза:')
display(Audio('/content/test.wav'))

# Автоматическое скачивание файлов модели
from google.colab import files
print('Скачивание my_voice.onnx, my_voice.onnx.json и test.wav...')
files.download('/content/my_voice.onnx')
files.download('/content/my_voice.onnx.json')
files.download('/content/test.wav')